# Ejercicio Flask API
Para este ejercicio tendrás que desplegar un modelo de machine learning en una API para su consumo en local. Entrenarás un modelo, lo guardarás entrenado, y desarrollarás una API que permita consumir dicho modelo desde cualquier otra tecnología.

**Se presenta el siguiente caso de uso**

Una empresa distribuidora de muebles del ámbito nacional pretende utilizar un modelo desarrollado por el departamento de analítica, con el que consiguen una predicción de las ventas a partir de los gastos en marketing de anuncios en televisión, radio y periódicos. Quieren incorporar estos datos dentro de su página web interna, donde comparten todo tipo de información relativa a resultados de la empresa, ventas, adquisiciones, etc... La web está desarrollada en AngularJS, mientras que el modelo se desarrolló en Python, por lo que precisamos de una interfaz de comunicación entre ambos sistemas.

El equipo de desarrollo necesita que implementes un microservicio para que ellos puedan consumir el modelo desde la propia web. El microservicio tiene que cumplir las siguientes características:
1. Ofrezca la predicción de ventas a partir de todos los valores de gastos en publicidad.
2. Podamos actualizar la base de datos con nuevos registros, una vez conozcamos los valores de venta reales.
3. Posibilidad de reentrenar el modelo con los nuevos registros.

**¿Qué no es necesario implementar, y se deja como ejercicio opcional?**
1. Por simplicidad del ejercicio, la base de datos con la que se trabaja es un CSV. Ingéstala en un SQLite, y actualiza la BD mediante el módilo de Python *sqlite3*.
2. Almacenar los datos por fecha. Lo ideal sería tener la fecha de almacenaje del dato, de cara a que no haya registros ducplicados por mes, y comprobando siempre que tenemos el dato lo más actualizado posible.
3. El modelo. En el momento de reentrenamiento, simplemente se entrena la regresión lineal con los últimos datos. No es necesario comprobar si los resultados son mejores o peores, tenemos missings u outliers...

**NOTA**: Cuentas con un script de Python (*model.py*) con el código de entrenamiento del modelo ya hecho, puesto que el desarrollo de un modelo de machine learning no es el objetivo del ejercicio, sino el diseño de una API con Flask

In [1]:
import requests

BASE_URL = 'http://127.0.0.1:5000'

### 1. `POST /predict` — Predicción de ventas

In [2]:
payload = {
    'TV':        230.1,
    'radio':      37.8,
    'newspaper':  69.2
}

response = requests.post(f'{BASE_URL}/predict', json=payload)
print(response.status_code)
print(response.json())

200
{'prediction': 20435.28}


In [3]:
# Comprobamos la validación de campos (falta 'newspaper' → debe devolver 400)
payload_incompleto = {'TV': 100.0, 'radio': 20.0}

response = requests.post(f'{BASE_URL}/predict', json=payload_incompleto)
print(response.status_code)  # 400
print(response.json())

400
{'error': "Faltan los campos: ['newspaper']"}


### 2. `POST /ingest` — Añadir nuevos registros a la BD

In [4]:
nuevo_registro = {
    'TV':        150.0,
    'radio':      30.0,
    'newspaper':  20.0,
    'sales':   16000.0
}

response = requests.post(f'{BASE_URL}/ingest', json=nuevo_registro)
print(response.status_code)  # 201
print(response.json())

201
{'message': 'Registro añadido correctamente'}


In [5]:
# Añadimos varios registros más para el reentrenamiento
nuevos = [
    {'TV': 200.0, 'radio': 40.0, 'newspaper': 10.0, 'sales': 19500.0},
    {'TV':  50.0, 'radio':  5.0, 'newspaper': 25.0, 'sales':  7000.0},
    {'TV': 300.0, 'radio': 50.0, 'newspaper':  5.0, 'sales': 27000.0},
]

for reg in nuevos:
    r = requests.post(f'{BASE_URL}/ingest', json=reg)
    print(r.status_code, r.json())

201 {'message': 'Registro añadido correctamente'}
201 {'message': 'Registro añadido correctamente'}
201 {'message': 'Registro añadido correctamente'}


### 3. `POST /retrain` — Reentrenar el modelo

In [6]:
response = requests.post(f'{BASE_URL}/retrain')
print(response.status_code)  # 200
print(response.json())       # Muestra n_samples usadas en el reentrenamiento

200
{'message': 'Modelo reentrenado correctamente', 'n_samples': 205}


In [7]:
# Predicción con el modelo ya reentrenado
payload = {'TV': 230.1, 'radio': 37.8, 'newspaper': 69.2}

response = requests.post(f'{BASE_URL}/predict', json=payload)
print(response.status_code)
print(response.json())

200
{'prediction': 20495.44}
